# 12 — Pull Cross-National Time Series Data Archive (CNTS)

**Source:** Cross-National Time Series Data Archive  
**Provider:** Arthur S. Banks / David Wilson (2021 Edition)  
**Access:** Commercial subscription (CQ Press / Databanks International)  
**Coverage:** Global, 1815–present (~200 countries)  
**Frequency:** Annual  

## What this notebook pulls

CNTS provides annual country-level time series covering **domestic conflict events**
(Banks' original 8-category schema) and a wide range of political and economic
structural indicators. It is used as the **primary dependent variable source** in
Medvedev et al. (2022) and as a predictor source in Redl & Hlatshwayo / IMF WP (2021).

### Banks' 8 domestic conflict event categories

| Variable | Description |
|---|---|
| `assas` | Assassinations (politically motivated) |
| `strikes` | General strikes |
| `guerrilla` | Guerrilla warfare incidents |
| `govtcrises` | Government crises |
| `purges` | Purges (systematic elimination of political opponents) |
| `riots` | Riots |
| `revols` | Revolutions |
| `protests` | Anti-government demonstrations / protests |

### Political structural indicators

| Variable | Description |
|---|---|
| `cabinetchanges` | Cabinet changes in the year |
| `constchanges` | Constitutional changes |
| `legiseffect` | Legislative effectiveness |
| `legisselect` | Legislative selection method |
| `executivechanges` | Changes in effective executive |
| `elections` | Legislative elections held (0/1) |

## Access note
CNTS is a **commercial dataset** available via CQ Press subscription
(https://www.cntsdata.com). It is typically distributed as a Stata `.dta` file
or an Excel workbook. This notebook expects the file to be placed locally:

```
cnts_files/cnts.dta     ← Stata format (preferred)
cnts_files/cnts.xlsx    ← Excel format (fallback)
cnts_files/cnts.csv     ← CSV format (fallback)
```

Set `CNTS_FILE_PATH` to override the default search path.

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
CNTS_FILE_PATH     — (optional) explicit path to CNTS data file
```

In [ ]:
import os
import pandas as pd
import pyreadstat
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Explicit override path, or search cnts_files/ directory
CNTS_FILE_PATH_ENV = os.getenv("CNTS_FILE_PATH", "")
CNTS_LOCAL_DIR     = Path("cnts_files")
CNTS_LOCAL_DTA     = CNTS_LOCAL_DIR / "cnts.dta"
CNTS_LOCAL_XLSX    = CNTS_LOCAL_DIR / "cnts.xlsx"
CNTS_LOCAL_CSV     = CNTS_LOCAL_DIR / "cnts.csv"

local_exists = (
    bool(CNTS_FILE_PATH_ENV)
    or CNTS_LOCAL_DTA.exists()
    or CNTS_LOCAL_XLSX.exists()
    or CNTS_LOCAL_CSV.exists()
)

print(f"Run date          : {RUN_DATE}")
print(f"CNTS_FILE_PATH set: {bool(CNTS_FILE_PATH_ENV)}")
print(f"Local files       : dta={CNTS_LOCAL_DTA.exists()}, xlsx={CNTS_LOCAL_XLSX.exists()}, csv={CNTS_LOCAL_CSV.exists()}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load CNTS data

Reads from explicit path, then tries `.dta` → `.xlsx` → `.csv` in `cnts_files/`.

In [ ]:
def load_cnts() -> pd.DataFrame | None:
    if CNTS_FILE_PATH_ENV:
        p = Path(CNTS_FILE_PATH_ENV)
        if not p.exists():
            print(f"CNTS_FILE_PATH set but file not found: {p}")
            return None
        print(f"Loading CNTS from CNTS_FILE_PATH: {p}")
        return _read_cnts_file(p)

    for path in [CNTS_LOCAL_DTA, CNTS_LOCAL_XLSX, CNTS_LOCAL_CSV]:
        if path.exists():
            print(f"Loading CNTS from: {path}")
            return _read_cnts_file(path)

    return None


def _read_cnts_file(path: Path) -> pd.DataFrame | None:
    suffix = path.suffix.lower()
    try:
        if suffix == ".dta":
            df, meta = pyreadstat.read_dta(str(path))
            return df
        elif suffix in (".xlsx", ".xls"):
            return pd.read_excel(str(path))
        elif suffix == ".csv":
            return pd.read_csv(str(path), low_memory=False)
        else:
            # Attempt Stata first, then CSV
            try:
                df, _ = pyreadstat.read_dta(str(path))
                return df
            except Exception:
                return pd.read_csv(str(path), low_memory=False)
    except Exception as e:
        print(f"  Failed to read {path}: {e}")
        if suffix == ".dta":
            try:
                print("  Retrying with pandas read_stata...")
                return pd.read_stata(str(path))
            except Exception as e2:
                print(f"  pandas read_stata also failed: {e2}")
        return None


df_cnts_raw = load_cnts()

if df_cnts_raw is None:
    print(
        "\nWARNING: CNTS data not available.\n"
        "  CNTS is a commercial dataset. Obtain it via:\n"
        "  https://www.cntsdata.com (CQ Press subscription)\n"
        "  Then place the file at one of:\n"
        f"    {CNTS_LOCAL_DTA}\n"
        f"    {CNTS_LOCAL_XLSX}\n"
        f"    {CNTS_LOCAL_CSV}\n"
        "  Or set CNTS_FILE_PATH to the explicit file path.\n"
    )
else:
    df_cnts_raw.columns = [c.lower().strip() for c in df_cnts_raw.columns]
    print(f"CNTS raw: {len(df_cnts_raw):,} rows | columns: {list(df_cnts_raw.columns)}")

## Schema — expected columns

CNTS column names have been stable across editions but may differ in casing
or abbreviation. We map to canonical names and validate presence.

In [ ]:
# Canonical name → possible source column names
CNTS_COL_ALIASES = {
    # Identifiers
    "ccode":           ["ccode", "cow", "wbcode", "countrycode", "code"],
    "country":         ["country", "countryname", "country_name", "statename"],
    "year":            ["year"],
    # Banks' 8 domestic conflict categories
    "assas":           ["assas", "assassinations", "assassin"],
    "strikes":         ["strikes", "gstrikes", "generalstrikes"],
    "guerrilla":       ["guerrilla", "guerrilla_warfare", "guerrillawf"],
    "govtcrises":      ["govtcrises", "govt_crises", "govcrises", "government_crises"],
    "purges":          ["purges"],
    "riots":           ["riots"],
    "revols":          ["revols", "revolutions", "revolution"],
    "protests":        ["protests", "antigovt", "demonstrations", "demo"],
    # Political structural indicators
    "cabinetchanges":  ["cabinetchanges", "cabinet_changes", "cabchanges"],
    "constchanges":    ["constchanges", "const_changes", "constitutional_changes"],
    "legiseffect":     ["legiseffect", "legislative_effectiveness", "legeffect"],
    "legisselect":     ["legisselect", "legislative_selection", "legselect"],
    "executivechanges":["executivechanges", "executive_changes", "execchanges"],
    "elections":       ["elections", "legelections", "leg_elections"],
}

BANKS_CONFLICT_COLS = ["assas", "strikes", "guerrilla", "govtcrises",
                       "purges", "riots", "revols", "protests"]
POLITICAL_COLS      = ["cabinetchanges", "constchanges", "legiseffect",
                       "legisselect", "executivechanges", "elections"]

if df_cnts_raw is not None:
    df_cnts = df_cnts_raw.copy()

    # Apply alias mapping
    rename_map = {}
    for canonical, aliases in CNTS_COL_ALIASES.items():
        if canonical in df_cnts.columns:
            continue
        for alias in aliases:
            if alias in df_cnts.columns:
                rename_map[alias] = canonical
                break
    if rename_map:
        df_cnts.rename(columns=rename_map, inplace=True)
        print(f"Renamed columns: {rename_map}")

    # Validate
    all_expected = list(CNTS_COL_ALIASES.keys())
    missing = [c for c in all_expected if c not in df_cnts.columns]
    found   = [c for c in all_expected if c in df_cnts.columns]
    if missing:
        print(f"Columns not found (may be named differently): {missing}")
    print(f"Columns confirmed present: {found}")
else:
    df_cnts = pd.DataFrame()

## Type casting and derived variables

In [ ]:
if not df_cnts.empty:
    # Integer identifiers
    for icol in ["ccode", "year"]:
        if icol in df_cnts.columns:
            df_cnts[icol] = pd.to_numeric(df_cnts[icol], errors="coerce").astype("Int64")

    # Count columns (Banks' 8 + political) — non-negative integers
    count_cols = BANKS_CONFLICT_COLS + POLITICAL_COLS
    for col in count_cols:
        if col in df_cnts.columns:
            df_cnts[col] = pd.to_numeric(df_cnts[col], errors="coerce").astype("Int64")

    # Derived: composite domestic conflict index (sum of Banks' 8 categories)
    present_conflict_cols = [c for c in BANKS_CONFLICT_COLS if c in df_cnts.columns]
    if present_conflict_cols:
        df_cnts["domestic_conflict_index"] = (
            df_cnts[present_conflict_cols]
            .apply(pd.to_numeric, errors="coerce")
            .sum(axis=1, skipna=True)
            .astype("Int64")
        )
        print(f"Derived domestic_conflict_index from: {present_conflict_cols}")

    # Derived: any_conflict flag
    if present_conflict_cols:
        df_cnts["any_conflict_event"] = (
            df_cnts["domestic_conflict_index"] > 0
        ).astype("Int8")

    if "year" in df_cnts.columns:
        print(f"Year range : {df_cnts['year'].min()}–{df_cnts['year'].max()}")
    if "country" in df_cnts.columns:
        print(f"Countries  : {df_cnts['country'].nunique()}")
    print(f"Shape      : {df_cnts.shape}")
    df_cnts.head(3)

In [ ]:
if not df_cnts.empty:
    write_parquet(df_cnts, f"raw/cnts/{RUN_DATE}/cnts_country_year.parquet")

## Conflict-event subset

Write a narrower table with just the Banks' 8 conflict categories + identifier columns
for lightweight joins in the panel-building step.

In [ ]:
if not df_cnts.empty:
    id_cols = [c for c in ["ccode", "country", "year"] if c in df_cnts.columns]
    conflict_subset_cols = id_cols + [
        c for c in BANKS_CONFLICT_COLS + ["domestic_conflict_index", "any_conflict_event"]
        if c in df_cnts.columns
    ]
    df_conflict = df_cnts[conflict_subset_cols].copy()
    print(f"Conflict subset: {df_conflict.shape}")
    write_parquet(df_conflict, f"raw/cnts/{RUN_DATE}/cnts_conflict_events.parquet")

## Summary

In [ ]:
print("=" * 55)
print("CNTS pull complete")
print("=" * 55)
if not df_cnts.empty:
    print(f"  Full panel rows : {len(df_cnts):,} country-years")
    if "country" in df_cnts.columns:
        print(f"  Countries       : {df_cnts['country'].nunique()}")
    if "year" in df_cnts.columns:
        print(f"  Year range      : {df_cnts['year'].min()}–{df_cnts['year'].max()}")
    print(f"  Columns         : {list(df_cnts.columns)}")
else:
    print("  No data loaded — see WARNING above for access instructions.")
print()
print("ADLS paths written:")
print(f"  raw/cnts/{RUN_DATE}/cnts_country_year.parquet")
print(f"  raw/cnts/{RUN_DATE}/cnts_conflict_events.parquet")